# v0.2: Passive Membrane Physics Tutorial

**Objective:** Understand membrane capacitance and resistance as electrical circuit elements, and validate numerical integration against exact exponential solutions.

**Scope:** Single soma passive membrane. Not biological validation; a computational scaffold for teaching.

**Reading time:** ~20 min. **Execution time:** ~5 sec.

## 1. Learning Objectives

By the end of this tutorial, you will:
- Understand passive membrane as a 1st-order ODE: C_m dV/dt = -g_L(V - E_L) + I_inj
- Know the steady-state voltage and time constant (tau = C_m / g_L)
- Validate forward-Euler integration against exact exponential solutions
- Recognize how timestep size affects numerical accuracy and stability
- Identify what passive membrane DOES NOT claim (LFP, CSD, EEG, etc.)

## 2. Biological/Computational Question

**Question:** When a neuron receives a small injected current, how does its voltage change over time?

**Biological context:**
- The soma is a roughly spherical cell body with a lipid membrane.
- The membrane has two key electrical properties: capacitance (stores charge) and conductance (leaks current).
- At rest (no current), the soma voltage approaches the leak reversal potential E_L (≈ −65 mV).
- Under steady current, the voltage exponentially relaxes to a new steady state.

**Why this matters:**
- Membrane dynamics are the foundation for understanding ion channels, synapses, and action potentials.
- We can solve the passive membrane equation exactly, so we can validate any numerical solver against it.
- Understanding this baseline helps identify when other mechanisms (Na/K channels, plasticity) become important.

## 3. Mathematical Glossary Flow

**Passive membrane equation (Hodgkin-Huxley reduced form):**

$$C_m \frac{dV}{dt} = -g_L(V - E_L) + I_{inj}$$

**Steady-state voltage (when dV/dt = 0):**

$$V_{ss} = E_L + \frac{I_{inj}}{g_L}$$

**Time constant (exponential decay rate):**

$$\tau = \frac{C_m}{g_L}$$

**Exact solution (constant I_inj):**

$$V(t) = V_{ss} + (V_{init} - V_{ss}) e^{-t/\tau}$$

**Forward-Euler step (numerical integration):**

$$V_{new} = V + \frac{dt}{C_m}[-g_L(V - E_L) + I_{inj}]$$

## 4. Units and Absolute Soma Convention

**Key design choice:** We use **absolute soma units**, not specific densities.

| Symbol | Unit | Range | Example |
|--------|------|-------|----------|
| V | mV | -100 to +50 | -65 mV (rest) |
| C_m | pF | 50–200 | 100 pF (typical soma) |
| g_L | nS | 0.1–5 | 1.0 nS (typical soma) |
| E_L | mV | -70 to -50 | -65 mV (leak reversal) |
| I_inj | pA | 0–200 | 10 pA (small current) |
| dt | ms | 0.01–1 | 1 ms (timestep) |
| tau | ms | 10–200 | tau = C_m / g_L = 100 ms |

**Why absolute units?** Mixing specific units (mS/cm²) with absolute current (pA) causes dimensional confusion. Using absolute soma values is clearer for teaching.

## 5. Simulation Block

In [ ]:
# Import the passive membrane module
from jbiophysic.passive_membrane import (
    PassiveMembraneParams,
    passive_membrane_simulate,
    steady_state_voltage,
    tau_membrane_ms,
    relaxation_curve,
)
import numpy as np

# Define parameters (typical soma values)
params = PassiveMembraneParams(
    C_m=100.0,   # pF
    g_L=1.0,     # nS
    E_L=-65.0,   # mV
)

# Calculate derived properties
tau = tau_membrane_ms(params.C_m, params.g_L)
print(f"Time constant tau = {tau:.1f} ms")

# Simulate with step current
I_inj = 10.0  # pA (inward, depolarizing)
V_ss = steady_state_voltage(params.E_L, params.g_L, I_inj)
print(f"Steady-state voltage V_ss = {V_ss:.2f} mV (depolarized by {V_ss - params.E_L:.1f} mV)")

V_trace = passive_membrane_simulate(
    V_init=-65.0,
    params=params,
    I_inj=I_inj,
    dt_ms=1.0,
    duration_ms=500.0,
)

print(f"Simulated {len(V_trace)-1} steps")
print(f"V(t=0) = {V_trace[0]:.2f} mV")
print(f"V(t=500 ms) = {V_trace[-1]:.3f} mV")
print(f"Expected V_ss = {V_ss:.3f} mV")

## 6. Diagnostics Block

In [ ]:
from jbiophysic.passive_membrane import membrane_potential_response
from jbiophysic.units import finite_value_check, magnitude_diagnostics

# Check that all values are finite
finite_check = finite_value_check(V_trace, "voltage")
print(f"All values finite: {finite_check['is_finite']}")

# Check magnitude is within reasonable bounds
mag_check = magnitude_diagnostics(V_trace, "voltage", expected_range=(-100, 0))
print(f"Voltage in range [-100, 0] mV: {mag_check['in_range']}")

# Characterize step response
resp = membrane_potential_response(I_step=I_inj, g_L=params.g_L, tau=tau)
print(f"\nStep response analysis:")
print(f"  Steady-state deflection: {resp['V_ss']:.2f} mV")
print(f"  Time to 50% of steady state: {resp['t_half']:.2f} ms")
print(f"  Time to 63.2% (1 tau): {resp['tau']:.2f} ms")

## 7. Parameter Sweep Block

In [ ]:
# Sweep over membrane capacitance
print("Effect of C_m on time constant:")
print(f"{'C_m (pF)':>10} | {'tau (ms)':>10}")
print("-" * 25)

for C_m in [50.0, 100.0, 200.0]:
    tau_test = tau_membrane_ms(C_m, params.g_L)
    print(f"{C_m:10.1f} | {tau_test:10.1f}")

# Sweep over conductance
print("\nEffect of g_L on time constant:")
print(f"{'g_L (nS)':>10} | {'tau (ms)':>10}")
print("-" * 25)

for g_L_test in [0.5, 1.0, 2.0]:
    tau_test = tau_membrane_ms(params.C_m, g_L_test)
    print(f"{g_L_test:10.1f} | {tau_test:10.1f}")

# Sweep over current
print("\nEffect of I_inj on steady-state voltage:")
print(f"{'I_inj (pA)':>12} | {'V_ss (mV)':>12}")
print("-" * 28)

for I_test in [0.0, 10.0, 50.0]:
    V_ss_test = steady_state_voltage(params.E_L, params.g_L, I_test)
    print(f"{I_test:12.1f} | {V_ss_test:12.3f}")

## 8. Wrong-Timestep and Wrong-Units Null Tests

In [ ]:
# Test: Large timestep produces unrealistic jumps
print("Effect of timestep size on step amplitude:")
print(f"{'dt (ms)':>10} | {'dV per step (mV)':>18}")
print("-" * 32)

for dt_test in [0.1, 1.0, 10.0, 100.0]:
    # Single step from V=-65, I_inj=10 pA
    dV = (1.0 / params.C_m) * (-params.g_L * (-65 - params.E_L) + I_inj) * dt_test
    print(f"{dt_test:10.1f} | {dV:18.4f}")

print("\n✓ Interpretation: Larger dt → larger single-step voltage changes")
print("  (Forward Euler is convergent but can have large truncation error per step)\n")

# Test: Units mismatch would produce nonsensical results
print("Hypothetical units mismatch (for illustration):")
print("If we accidentally used g_L in mS/cm² (0.1) instead of nS (1.0):")
I_test = 10.0  # pA
g_L_wrong = 0.1  # mS/cm² [WRONG UNITS]
V_ss_wrong = params.E_L + I_test / g_L_wrong  # Would give nonsensical mV
print(f"  Wrong: V_ss = {V_ss_wrong:.1f} mV (unphysical!)")
print(f"  Right: V_ss = {V_ss:.1f} mV (reasonable)")

## 9. Manifest and Report Block

In [ ]:
# Generate a report-style manifest
import json

manifest = {
    "chapter": "v0.2",
    "tutorial": "passive_membrane_physics",
    "parameters": {
        "C_m_pF": params.C_m,
        "g_L_nS": params.g_L,
        "E_L_mV": params.E_L,
        "I_inj_pA": I_inj,
        "dt_ms": 1.0,
        "duration_ms": 500.0,
    },
    "derived": {
        "tau_ms": tau,
        "V_ss_mV": V_ss,
        "input_resistance_mohm": 1000.0 / params.g_L,
    },
    "simulation_outcome": {
        "V_init_mV": float(V_trace[0]),
        "V_final_mV": float(V_trace[-1]),
        "n_steps": len(V_trace) - 1,
        "all_finite": bool(np.all(np.isfinite(V_trace))),
        "error_from_steady_state_mV": float(abs(V_trace[-1] - V_ss)),
    },
    "truth_status": "truth_safe_unverified",
    "claim_type": "computational_scaffold / teaching_template",
}

print(json.dumps(manifest, indent=2))

## 10. Interpretation

In [ ]:
# Validate numerical solution against exact analytical solution
t_analytical = np.linspace(0, 500, len(V_trace))
V_analytical = relaxation_curve(t_analytical, V_init=-65.0, V_ss=V_ss, tau=tau)

max_error = np.max(np.abs(V_trace - V_analytical))
mean_error = np.mean(np.abs(V_trace - V_analytical))

print("Numerical vs. Analytical Comparison:")
print(f"  Max error: {max_error:.4f} mV")
print(f"  Mean error: {mean_error:.6f} mV")
print(f"  Error relative to steady-state deflection: {max_error / (V_ss - (-65.0)) * 100:.2f}%")
print(f"\n✓ Forward Euler with dt=1 ms converges well to exact solution.")
print(f"  This validates our simulator against ground truth.")

## 11. Failure Modes

**When passive membrane simulation fails or gives misleading results:**

1. **Very large timestep (dt > 2*tau):** Euler method becomes numerically unstable. Single steps can overshoot or oscillate.
   - *Example:* dt = 300 ms with tau = 100 ms → unrealistic jumps.
   - *Fix:* Use dt < 0.1 * tau or switch to implicit/RK4 integrator.

2. **Mismatched units:** Using specific conductance (mS/cm²) with absolute current (pA) → dimensional confusion.
   - *Example:* g_L_specific = 0.1 mS/cm² ≠ g_L = 1 nS (absolute soma).
   - *Fix:* Ensure all parameters are absolute soma units (pF, nS, pA).

3. **No ion channels:** Passive membrane saturates at V_ss; real neurons spike via Na/K channels.
   - *Example:* I_inj = 100 pA depolarizes to V_ss, no action potential.
   - *Fix:* Add Hodgkin-Huxley conductances (v0.3).

4. **Single compartment:** Real soma has complex morphology, axon, dendrites.
   - *Example:* Our V is soma average; no spatial V_m gradients.
   - *Fix:* Use multi-compartment models (v0.4+).

5. **No synaptic input:** Passive membrane assumes only injected current.
   - *Example:* Real neurons receive thousands of synaptic events.
   - *Fix:* Add synaptic conductances (v0.4).

## 12. Exercises

**Exercise 1:** Derive the steady-state voltage formula.
- At equilibrium (dV/dt = 0), what is the relationship between V_ss, E_L, I_inj, and g_L?
- Show your work.

**Exercise 2:** Compute time to half-maximal deflection.
- Given tau = 100 ms, how long does it take to reach 50% of the way to V_ss?
- (Hint: solve V(t) = V_ss - 0.5*(V_ss - V_init) for t.)

**Exercise 3:** Compare numerical and analytical at t = tau.
- Run passive_membrane_simulate and relaxation_curve at t = tau.
- Are they within 1% of each other?

**Exercise 4:** Explain the large-timestep artifact.
- Why does dt = 100 ms produce a much larger single-step voltage change than dt = 1 ms?
- Is this a failure of the simulator or a limitation of the Euler method?

**Exercise 5:** What is NOT established by passive membrane voltage?
- If V changes by 10 mV, does that prove there is extracellular LFP? Why or why not?
- What additional information is needed to claim an extracellular field?

**Exercise 6:** Design a manifest.
- What 5 key pieces of metadata would you include to declare passive membrane current as a source for jaxfne forward-field modeling?
- (See docs/v0_2_jaxfne_bridge.md for hints.)

## 13. What This Tutorial DOES NOT Claim

❌ **Not biological calibration.** This is a teaching model; passive membrane dynamics are illustrative but not validated against real neurons.

❌ **Not extracellular field (LFP, CSD, EEG, MEG).** Transmembrane voltage ≠ extracellular field. Field requires source-field-probe coupling via jaxfne (future).

❌ **Not sufficient for spike generation.** Action potentials require voltage-gated Na/K channels (v0.3).

❌ **Not whole-neuron behavior.** Single soma; no axon, dendrites, or morphology (v0.4+).

❌ **Not network behavior.** No synapses or interactions (v0.4+).

❌ **Not frequency/oscillation analysis.** No inherent oscillations in passive membrane; frequencies come from ion channels or network.

✓ **IS:** A numerically validated ground truth for teaching membrane physics, unit conventions, and integration methods.